# This file is about cleaning the 'actor' column of traces250102_clean.csv
The output of this file is the input for the anonymizing.py

## Libraries

In [2]:
# ---
# jupyter:
#   jupytext:
#     formats: ipynb,py:light
#     text_representation:
#       extension: .py
#       format_name: light
#       format_version: '1.5'
#       jupytext_version: 1.17.0
#   kernelspec:
#     display_name: venv_jupyter
#     language: python
#     name: venv_jupyter
# ---

# # Regarder les identifiants avant anonymisation

# import les bibliothèque
import sys
sys.path.append('../') # ces deux lignes permettent au notebook de trouver le chemin jusqu'au code source contenu dans 'src'
sys.path.append('../src/') # Yvan dit que c'est absurde, mais sans ça ne marche pas
from src import * # on importe ce qui se trouve dans 'src'
import pandas as pd
#from src import script_initialisation, clean_corrupted_data *not used*
from src.data import cleaning
from src.features import utils
import re
import time
from src.data.constants import INTERIM_DATA_DIR

# mise à jour Automatiquement
get_ipython().run_line_magic('load_ext', 'autoreload') # pour mise à jour 
get_ipython().run_line_magic('autoreload', '2') # pour mise à jour

## Loading Dataframe

In [6]:
# Read dataframe from load_clean_data() which cleans: timestamp, time_delta, session.duration

df = pd.read_csv(r'D:\univ\analyse-des-traces\New src\data\interim\traces250102_clean.csv')

print(df.head())

print(df.columns)

   research_usage  result                  _id.$oid  \
0             1.0     NaN  675aa485a7dff6d2d4811b45   
1             1.0     NaN  675aa4cea7dff6d2d4811b4f   
2             1.0     NaN  675aa4cea7dff6d2d4811b50   
3             1.0     NaN  671b54aabd5a98b8f9dc8c14   
4             1.0     NaN  671b54f6bd5a98b8f9dc8c45   

                    timestamp.$date              stored.$date  \
0  2024-12-12 09:53:24.624000+00:00  2024-11-27T10:57:29.801Z   
1  2024-12-12 09:53:25.008000+00:00  2024-11-27T10:57:29.801Z   
2  2024-12-12 09:54:38.793000+00:00  2024-11-27T10:57:29.801Z   
3  2024-10-25 10:19:54.344000+00:00  2024-08-29T07:34:11.687Z   
4  2024-10-25 10:19:54.583000+00:00  2024-08-29T07:34:11.687Z   

             actor actor.objectType           verb  \
0  abaly.oura.etu/            Agent  Session.Start   
1  abaly.oura.etu/            Agent      File.Open   
2  abaly.oura.etu/            Agent    Session.End   
3  abaly.oura.etu/            Agent  Session.Start   
4  abaly

## Cleaning process

- Change the format of time and date
- Keep only prenom.nom.etu of one student in actor column
- Clean 'P_codeState' and 'F_codeState' columns

### Cleaning timestamp, time_delta, session.duration

In [ ]:
from typing import Optional

def clean_time_date(df: pd.DataFrame) -> None:
    """
    clean log data with time/duration conversions.
    
    Args:
        df: .
        interim_data_dir: Directory path for the CSV file. If None, uses INTERIM_DATA_DIR.
        parse_dates: If True, converts timestamp and duration columns.
    
    Returns:
        Cleaned DataFrame with proper datetime/duration types.
    """
    # Path handling
    base_path = interim_data_dir if interim_data_dir is not None else INTERIM_DATA_DIR
    file_path = f"{base_path}{file_name}"
    
    # Optimized column loading with dtype inference
    logs = pd.read_csv(
        file_path,
        keep_default_na=False,
        dtype={'session.id': 'string'},  # More memory-efficient than str
        parse_dates=['timestamp.$date'] if parse_dates else False,
        infer_datetime_format=True  # Faster parsing
    )
    
    if parse_dates:
        # Vectorized datetime conversion (faster than column-wise)
        time_cols = {
            'timestamp.$date': 'datetime64[ns]',
            'time_delta': 'timedelta64[ns]',
            'session.duration': 'timedelta64[ns]'
        }
        
        logs = logs.astype(time_cols)
    
    # Reset index without assignment warning
    return logs.reset_index(drop=True, copy=False)

NameError: name 'pd' is not defined

### Cleaning actor column

We want to clean the 'actor' column where present the prenom.nom.etu of students. The problems which we have in this column are:
- Remove '/' from the end of each name
- Remove the binom student, we just want the first student
- Remove '@univ-lille.fr/' part from the name
- Remove all invalid names like nebut or luc

At the end, the column actor should only include the name of student with the 'prenom.nom.etu' format

In [3]:
# Define the regex pattern

pattern = r'^[a-zA-Z\-]+\.[a-zA-Z0-9\-]+\.etu$'
invalid_names = []

# Apply a cleaning function
def clean_actor(name):
    """
    clean actors with this pattern 'prenom.nom.etu'.

    Args:
        name : A row of DataFrame.

    Returns:
        the cleaned name of actor
    """
     
    #print(name) this is for test
    if isinstance(name, str):
        # remove '/'
        name = name.split('/')

        # remove ' '
        if '' in name : name.remove('')

        # keep only one student
        name = name[0]
    
        #print(name) this is for test
        
        # check pattern prenom.nom.etu
        if re.match(pattern, name):
            return name  # valid, keep as is
        
        elif '@' in name:  # maybe correctable
            match = re.match(r'^([a-zA-Z\-]+\.[a-zA-Z0-9\-]+\.etu)@', name)

            if match:
                return match.group(1)  # extract clean part
        else: 
            invalid_names.append(name)
            
    return None  # discard bad names


## Testing process on a column

In [4]:
# Test function of clean_actor(name)
def test_on_dataframe(df: pd.DataFrame, column: str) -> int:
    """
    Counts the number of values in a DataFrame column that do NOT match
    the pattern 'prenom.nom.etu'.

    Args:
        df (pd.DataFrame): The input DataFrame.
        column (str): The name of the column to check.

    Returns:
        int: Number of values not matching the expected pattern.
    """

    pattern = r'^[a-zA-Z\-]+\.[a-zA-Z0-9\-]+\.etu$'
    non_matching_indices = df[~df[column].astype(str).str.strip().str.fullmatch(pattern)].index.tolist()
    
    return df[column].apply(lambda x: not re.fullmatch(pattern, str(x).strip())).sum(), non_matching_indices
   

## Main function

In [ ]:
# main function to start the process
def main():
    """
    Start the process cleaning and testing.

    Args:
        None

    Returns:
        None
    """
    
    # Before cleaning function
    print("Before applying clean_actor() on df:")
    print(df['actor'])

    # Start test to get the unmatch actors 
    print("\n Test process...")
    # calculate the time spending
    start_time = time.time()
    not_matching, _  = test_on_dataframe(df,'actor')
    end_time = time.time()
    elapsed_time = end_time - start_time 
    print(f"Test done! in {elapsed_time:.4f} seconds")

    print(f"The total number of not matching : {not_matching} \n")

    # Apply cleaning function
    print("Apply cleaning function...")
    # calculate the time spending
    start_time = time.time()
    df['clean_actor'] = df['actor'].apply(clean_actor)
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Apply done! in {elapsed_time:.4f} seconds \n")

    # After cleaning function
    print("After applying clean_actor() on df")
    print(df[['actor','clean_actor']])

    # Start test to get the unmatch actors
    print("Test process...")
    # calculate the time spending
    start_time = time.time()
    not_matching, index_not_matching = test_on_dataframe(df,'clean_actor')
    end_time = time.time()
    elapsed_time = end_time - start_time
    print(f"Test done! in {elapsed_time:.4f} seconds")
    print('\n')

    print(f"The total number of not matching : {not_matching} ")
    
    for row in index_not_matching :
        print(df.loc[row, 'actor'])

In [6]:
main()

Before applying clean_actor() on df:
0                               abaly.oura.etu/
1                               abaly.oura.etu/
2                               abaly.oura.etu/
3                               abaly.oura.etu/
4                               abaly.oura.etu/
                          ...                  
306941    zacharie.deroo.etu/noe.carpentier.etu
306942    zacharie.deroo.etu/noe.carpentier.etu
306943    zacharie.deroo.etu/noe.carpentier.etu
306944    zacharie.deroo.etu/noe.carpentier.etu
306945    zacharie.deroo.etu/noe.carpentier.etu
Name: actor, Length: 306946, dtype: object

 Test process...
Test done! in 1.0203 seconds
The total number of not matching : 306946 

Apply cleaning function...
Apply done! in 0.5661 seconds 

After applying clean_actor() on df
                                        actor         clean_actor
0                             abaly.oura.etu/      abaly.oura.etu
1                             abaly.oura.etu/      abaly.oura.etu
2        

In [8]:
df

,research_usage,result,_id.$oid,timestamp.$date,stored.$date,actor,actor.objectType,verb,object,object.objectType,...,filename,F_codeState,Debug_TimeStampEnd,Debug_TimeStampActions,tests,function,time_delta,session.duration,seance,clean_actor
0,1.0,,675aa485a7dff6d2d4811b45,2024-12-12 09:53:24.624000+00:00,2024-11-27T10:57:29.801Z,abaly.oura.etu/,Agent,Session.Start,https://www.cristal.univ-lille.fr/objects/Session,Activity,...,,,,,,,7 days 00:05:05.313000,NaT,CTP,abaly.oura.etu
1,1.0,,675aa4cea7dff6d2d4811b4f,2024-12-12 09:53:25.008000+00:00,2024-11-27T10:57:29.801Z,abaly.oura.etu/,Agent,File.Open,https://www.cristal.univ-lille.fr/objects/File,Activity,...,/home/l1/abaly.oura.etu/Downloads/puissance4.py,"# Compléter ici (noms, groupe, contenu fichier...",,,,,0 days 00:00:00.384000,NaT,CTP,abaly.oura.etu
2,1.0,,675aa4cea7dff6d2d4811b50,2024-12-12 09:54:38.793000+00:00,2024-11-27T10:57:29.801Z,abaly.oura.etu/,Agent,Session.End,https://www.cristal.univ-lille.fr/objects/Session,Activity,...,,,,,,,0 days 00:01:13.785000,0 days 00:01:14.169000,CTP,abaly.oura.etu
3,1.0,,671b54aabd5a98b8f9dc8c14,2024-10-25 10:19:54.344000+00:00,2024-08-29T07:34:11.687Z,abaly.oura.etu/,Agent,Session.Start,https://www.cristal.univ-lille.fr/objects/Session,Activity,...,,,,,,,1 days 00:21:30.977000,NaT,semaine_8,abaly.oura.etu
4,1.0,,671b54f6bd5a98b8f9dc8c45,2024-10-25 10:19:54.583000+00:00,2024-08-29T07:34:11.687Z,abaly.oura.etu/,Agent,File.Open,https://www.cristal.univ-lille.fr/objects/File,Activity,...,/home/l1/abaly.oura.etu/TDM 8/devinette.py,# L1 MI S1 - Prog\n# TP exos de manipulation -...,,,,,0 days 00:00:00.239000,NaT,semaine_8,abaly.oura.etu
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
306941,1.0,,66d85000bd5a98b8f9d9278d,2024-09-04 14:16:21.990000+00:00,2024-08-29T07:34:11.687Z,zacharie.deroo.etu/noe.carpentier.etu,Agent,Run.Program,https://www.cristal.univ-lille.fr/objects/Program,Activity,...,,,,,,,0 days 00:00:53.790000,NaT,semaine_1,zacharie.deroo.etu
306942,1.0,,66d85056bd5a98b8f9d927af,2024-09-04 14:18:08.724000+00:00,2024-08-29T07:34:11.687Z,zacharie.deroo.etu/noe.carpentier.etu,Agent,Run.Program,https://www.cristal.univ-lille.fr/objects/Program,Activity,...,,,,,,,0 days 00:01:46.734000,NaT,semaine_1,zacharie.deroo.etu
306943,,,,2024-09-04 14:18:08.725000+00:00,,zacharie.deroo.etu/noe.carpentier.etu,,Session.End,,,...,,,,,,,0 days 00:00:00.001000,0 days 00:22:32.511000,semaine_1,zacharie.deroo.etu
306944,1.0,,66d845a3bd5a98b8f9d92625,2024-09-04 13:29:54.300000+00:00,2024-08-29T07:34:11.687Z,zacharie.deroo.etu/noe.carpentier.etu,Agent,Session.Start,https://www.cristal.univ-lille.fr/objects/Session,Activity,...,,,,,,,0 days 00:00:00,NaT,semaine_1,zacharie.deroo.etu


## Save cleaned data into a CSV file

In [43]:
df.to_csv(INTERIM_DATA_DIR + "acteurs_corriges_2425.csv")

## Check Dataframe

In [44]:
df_new = pd.read_csv(INTERIM_DATA_DIR + 'acteurs_corriges_2425.csv')

print(df_new.head())

   Unnamed: 0  research_usage  result                  _id.$oid  \
0           0             1.0     NaN  675aa485a7dff6d2d4811b45   
1           1             1.0     NaN  675aa4cea7dff6d2d4811b4f   
2           2             1.0     NaN  675aa4cea7dff6d2d4811b50   
3           3             1.0     NaN  671b54aabd5a98b8f9dc8c14   
4           4             1.0     NaN  671b54f6bd5a98b8f9dc8c45   

                    timestamp.$date              stored.$date  \
0  2024-12-12 09:53:24.624000+00:00  2024-11-27T10:57:29.801Z   
1  2024-12-12 09:53:25.008000+00:00  2024-11-27T10:57:29.801Z   
2  2024-12-12 09:54:38.793000+00:00  2024-11-27T10:57:29.801Z   
3  2024-10-25 10:19:54.344000+00:00  2024-08-29T07:34:11.687Z   
4  2024-10-25 10:19:54.583000+00:00  2024-08-29T07:34:11.687Z   

             actor actor.objectType           verb  \
0  abaly.oura.etu/            Agent  Session.Start   
1  abaly.oura.etu/            Agent      File.Open   
2  abaly.oura.etu/            Agent    Sessi

In [45]:
df_new.head()

,Unnamed: 0,research_usage,result,_id.$oid,timestamp.$date,stored.$date,actor,actor.objectType,verb,object,...,filename,F_codeState,Debug_TimeStampEnd,Debug_TimeStampActions,tests,function,time_delta,session.duration,seance,clean_actor
0,0,1.0,NaN,675aa485a7dff6d2d4811b45,2024-12-12 09:53:24.624000+00:00,2024-11-27T10:57:29.801Z,abaly.oura.etu/,Agent,Session.Start,https://www.cristal.univ-lille.fr/objects/Session,...,NaN,NaN,NaN,NaN,NaN,NaN,7 days 00:05:05.313000,NaN,CTP,abaly.oura.etu
1,1,1.0,NaN,675aa4cea7dff6d2d4811b4f,2024-12-12 09:53:25.008000+00:00,2024-11-27T10:57:29.801Z,abaly.oura.etu/,Agent,File.Open,https://www.cristal.univ-lille.fr/objects/File,...,/home/l1/abaly.oura.etu/Downloads/puissance4.py,"# Compléter ici (noms, groupe, contenu fichier...",NaN,NaN,NaN,NaN,0 days 00:00:00.384000,NaN,CTP,abaly.oura.etu
2,2,1.0,NaN,675aa4cea7dff6d2d4811b50,2024-12-12 09:54:38.793000+00:00,2024-11-27T10:57:29.801Z,abaly.oura.etu/,Agent,Session.End,https://www.cristal.univ-lille.fr/objects/Session,...,NaN,NaN,NaN,NaN,NaN,NaN,0 days 00:01:13.785000,0 days 00:01:14.169000,CTP,abaly.oura.etu
3,3,1.0,NaN,671b54aabd5a98b8f9dc8c14,2024-10-25 10:19:54.344000+00:00,2024-08-29T07:34:11.687Z,abaly.oura.etu/,Agent,Session.Start,https://www.cristal.univ-lille.fr/objects/Session,...,NaN,NaN,NaN,NaN,NaN,NaN,1 days 00:21:30.977000,NaN,semaine_8,abaly.oura.etu
4,4,1.0,NaN,671b54f6bd5a98b8f9dc8c45,2024-10-25 10:19:54.583000+00:00,2024-08-29T07:34:11.687Z,abaly.oura.etu/,Agent,File.Open,https://www.cristal.univ-lille.fr/objects/File,...,/home/l1/abaly.oura.etu/TDM 8/devinette.py,# L1 MI S1 - Prog\n# TP exos de manipulation -...,NaN,NaN,NaN,NaN,0 days 00:00:00.239000,NaN,semaine_8,abaly.oura.etu
